# Fine-Tuning TabICL on Imbalanced Classification: SFT vs DPO

This notebook demonstrates how to fine-tune **TabICL** — a pretrained in-context
learning model for tabular data — on a **strongly imbalanced** 3-class synthetic
dataset (class weights ≈ [1 %, 1 %, 98 %]).  We compare five fine-tuning
configurations across two training paradigms:

| Run | Loss function | Negative-sampling strategy |
|---|---|---|
| **SFT** | Cross-entropy | — |
| **DPO-random** | DPO | Uniformly random wrong class |
| **DPO-majority** | DPO | Always the majority class (class 2) |
| **DPO-hard** | DPO | Highest-probability wrong class from the frozen reference model |
| **DPO-confusion** | DPO | Most historically confused class (confusion-matrix pass) |

---

## Why DPO for Imbalanced Classification?

Standard supervised fine-tuning (SFT) with cross-entropy optimises the log-likelihood
of the correct class.  On a dataset where 98 % of samples belong to class 2, the model
can achieve misleadingly high *accuracy* simply by always predicting class 2, while
completely ignoring the two minority classes.

**Direct Preference Optimisation (DPO)** augments each training step with an explicit
*rejected* (negative) label and a frozen *reference model* (the pretrained checkpoint
before any fine-tuning).  The DPO loss — parameterised by a temperature β — penalises
the trainable model for assigning high probability to the rejected class *relative to
the reference*, providing a targeted signal to pull probability mass away from specific
wrong predictions.  Combined with minority-aware negative-sampling strategies, this can
significantly improve precision, recall, and F1 on the under-represented classes.

---
## 0. Shared Configuration

All hyperparameters are defined here as plain Python variables.  They are interpolated
into the CLI commands in the cells below, ensuring every run uses **identical** data,
optimiser, and split settings so comparisons are fair.

In [ ]:
# ── Dataset ─────────────────────────────────────────────────────────────────
N_SAMPLES            = 1000
N_CLASSES            = 3
N_FEATURES           = 20
N_INFORMATIVE        = 4
WEIGHTS              = [0.01, 0.01, 0.98]   # strongly imbalanced
DATASET_RANDOM_STATE = 0

# ── Splits ──────────────────────────────────────────────────────────────────
VAL_SIZE  = 0.15
TEST_SIZE = 0.15

# ── Training ────────────────────────────────────────────────────────────────
EPOCHS           = 10
BATCH_SIZE       = 16
CONTEXT_FRACTION = 0.7
LR               = 1e-5
WEIGHT_DECAY     = 0.0
GRADIENT_CLIP    = 1.0
SEED             = 42
LOG_EVERY        = 5
VAL_EVERY        = 1

# ── DPO ─────────────────────────────────────────────────────────────────────
BETA = 0.1   # preference temperature (higher → stricter alignment to reference)

# ── Output ──────────────────────────────────────────────────────────────────
RUNS_DIR     = 'runs/demo'
DATASET_PATH = f'{RUNS_DIR}/dataset.npz'

# ── Pre-build CLI argument strings ────────────────────────────────────────────
WEIGHTS_STR = ' '.join(str(w) for w in WEIGHTS)

# Arguments for make_data.py — controls dataset generation
MAKE_DATA_ARGS = (
    f'--n-samples {N_SAMPLES} --n-classes {N_CLASSES} '
    f'--n-features {N_FEATURES} --n-informative {N_INFORMATIVE} '
    f'--weights {WEIGHTS_STR} '
    f'--dataset-random-state {DATASET_RANDOM_STATE} '
    f'--val-size {VAL_SIZE} --test-size {TEST_SIZE} '
    f'--seed {SEED} '
    f'--output-path {DATASET_PATH}'
)

# Arguments shared by sft.py and dpo.py — dataset loaded from disk
COMMON_ARGS = (
    f'--dataset-path {DATASET_PATH} '
    f'--epochs {EPOCHS} --lr {LR} --batch-size {BATCH_SIZE} '
    f'--context-fraction {CONTEXT_FRACTION} --weight-decay {WEIGHT_DECAY} '
    f'--gradient-clip {GRADIENT_CLIP} --seed {SEED} '
    f'--val-size {VAL_SIZE} --test-size {TEST_SIZE} '
    f'--log-every {LOG_EVERY} --val-every {VAL_EVERY} '
    f'--save-ckpt'
)
print('make_data CLI arguments:')
print(MAKE_DATA_ARGS)
print()
print('Shared training CLI arguments:')
print(COMMON_ARGS)

---
## 1. Generate Dataset

Before any fine-tuning we generate the dataset **once** with `make_data.py` and save
it as a `.npz` file.  Both `sft.py` and `dpo.py` will load this file via
`--dataset-path`, guaranteeing that all runs train and are evaluated on **exactly the
same splits** regardless of environment or random-state quirks.

In [ ]:
import os
os.makedirs(RUNS_DIR, exist_ok=True)

make_data_cmd = f'uv run python -m tabicl.finetune.make_data {MAKE_DATA_ARGS}'
print(make_data_cmd)
!{make_data_cmd}

---
## 2. Supervised Fine-Tuning (SFT) — Baseline

SFT fine-tunes TabICL with standard **cross-entropy** loss computed on the query
samples of each in-context episode.  At every gradient step the dataloader re-samples
a random train/test split of the training set, exposing the model to diverse episode
configurations and acting as implicit data augmentation.

This serves as our **baseline**: the pretrained weights are adapted toward the target
distribution, but the loss gives no explicit signal about minority-class separation.

In [ ]:
sft_cmd = (
    f'uv run python -m tabicl.finetune.sft {COMMON_ARGS} '
    f'--output-dir {RUNS_DIR}/sft'
)
print(sft_cmd)
!{sft_cmd}

---
## 3. Direct Preference Optimisation (DPO)

DPO replaces cross-entropy with a **preference-ranking loss**.  For each query sample
the model is given:

- **Chosen** (positive) label → the ground-truth class
- **Rejected** (negative) label → a wrong class chosen by the negative-sampling strategy

The DPO loss with temperature β then maximises:

$$\mathcal{L}_{\mathrm{DPO}} = -\log\sigma\!\left(\beta\cdot\bigl[(\log\pi_\theta(y^+|x) - \log\pi_\theta(y^-|x)) - (\log\pi_{\mathrm{ref}}(y^+|x) - \log\pi_{\mathrm{ref}}(y^-|x))\bigr]\right)$$

where $\pi_\theta$ is the trainable model and $\pi_{\mathrm{ref}}$ is the **frozen**
pretrained checkpoint.  This simultaneously pushes probability toward the correct class
and *away from* the rejected class, relative to what the reference model already knows.

We compare **four negative-sampling strategies**:

| Strategy | Description |
|---|---|
| `random` | Uniformly random wrong class — simple, unbiased baseline |
| `majority` | Always the majority class (class 2, weight 98 %) — directly penalises majority-class bias |
| `hard` | Highest-probability wrong class from the reference model — strongest gradient signal |
| `confusion` | Most historically confused class per true label — dataset-level structure |


### 2.1 DPO — Random Negative Sampling

The simplest DPO baseline: for each query sample we pick a **uniformly random**
incorrect class as the rejected label.  This is class-distribution-agnostic and
provides a general regularisation signal without any knowledge of which classes are
confusable or dominant.

In [ ]:
dpo_random_cmd = (
    f'uv run python -m tabicl.finetune.dpo {COMMON_ARGS} '
    f'--beta {BETA} --preference-generator random '
    f'--output-dir {RUNS_DIR}/dpo_random'
)
print(dpo_random_cmd)
!{dpo_random_cmd}

### 2.2 DPO — Majority Class as Negative

This strategy always sets the rejected label to the **majority class** (class 2,
weight ≈ 98 %).  Every gradient step explicitly penalises the model for predicting
the dominant class, directly counteracting the majority-collapse bias that plagues
imbalanced classification.  For samples that *are* the majority class, the generator
falls back to the next most frequent class.

In [ ]:
dpo_majority_cmd = (
    f'uv run python -m tabicl.finetune.dpo {COMMON_ARGS} '
    f'--beta {BETA} --preference-generator majority '
    f'--output-dir {RUNS_DIR}/dpo_majority'
)
print(dpo_majority_cmd)
!{dpo_majority_cmd}

### 2.3 DPO — Hard Negative Mining

Uses the **frozen reference model** to identify the most confusable incorrect class:
for each query sample, the rejected label is whichever wrong class received the
*highest softmax probability* from the reference model.  These are the hardest
possible negatives — the errors the pretrained model is most likely to make — and
therefore provide the strongest gradient signal for correcting its biggest mistakes.

In [ ]:
dpo_hard_cmd = (
    f'uv run python -m tabicl.finetune.dpo {COMMON_ARGS} '
    f'--beta {BETA} --preference-generator hard '
    f'--output-dir {RUNS_DIR}/dpo_hard'
)
print(dpo_hard_cmd)
!{dpo_hard_cmd}

### 2.4 DPO — Confusion-Set Pairing

Before training begins, a single in-context inference pass on the training set
computes a **confusion matrix** using the pretrained reference model.  The rejected
label for each true class is then *fixed* to the class it is most frequently confused
*with* in that matrix.  Unlike hard-negative mining (which is instance-level), this
is a **dataset-level** strategy: the same rejected class is used for all samples of a
given true class, providing a stable and consistent signal throughout training.

In [ ]:
dpo_confusion_cmd = (
    f'uv run python -m tabicl.finetune.dpo {COMMON_ARGS} '
    f'--beta {BETA} --preference-generator confusion '
    f'--output-dir {RUNS_DIR}/dpo_confusion'
)
print(dpo_confusion_cmd)
!{dpo_confusion_cmd}

---
## 4. Comparing Results

We now load the TensorBoard event files written during training and plot learning
curves for all five fine-tuning runs.  We then evaluate every model — including two
**pretrained baselines** (vanilla TabICL and Neuralk NICL) — on the held-out test set.

The same **colour code** is used across every figure:

| Run | Colour |
|---|---|
| Vanilla TabICL | Dark grey |
| Neuralk NICL | Coral |
| SFT | Blue |
| DPO-random | Orange |
| DPO-majority | Green |
| DPO-hard | Red |
| DPO-confusion | Purple |

All classification metrics (precision, recall, F1) are **macro-averaged** across
the three classes, giving equal weight to the two minority classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── Colour palette for fine-tuned runs (used in training / val curves) ────────
COLORS = {
    'SFT':           '#4C72B0',   # blue
    'DPO-random':    '#DD8452',   # orange
    'DPO-majority':  '#55A868',   # green
    'DPO-hard':      '#C44E52',   # red
    'DPO-confusion': '#8172B2',   # purple
}

# ── Extended palette including pretrained baselines (for test bar charts) ─────
ALL_COLORS = {
    'Vanilla TabICL': '#555555',   # dark grey
    'Neuralk NICL':   '#E76F51',   # coral
    **COLORS,
}

# ── Run name → TensorBoard log directory ─────────────────────────────────────
RUNS = {
    'SFT':           f'{RUNS_DIR}/sft',
    'DPO-random':    f'{RUNS_DIR}/dpo_random',
    'DPO-majority':  f'{RUNS_DIR}/dpo_majority',
    'DPO-hard':      f'{RUNS_DIR}/dpo_hard',
    'DPO-confusion': f'{RUNS_DIR}/dpo_confusion',
}

def load_scalars(log_dir, tag):
    """Return (steps, values) for *tag* from a TensorBoard log directory."""
    ea = EventAccumulator(log_dir, size_guidance={'scalars': 0})
    ea.Reload()
    try:
        events = ea.Scalars(tag)
        return [e.step for e in events], [e.value for e in events]
    except KeyError:
        return [], []

legend_handles = [mpatches.Patch(color=c, label=n) for n, c in COLORS.items()]
print('Setup complete.')

### 3.1 Training Curves

Metrics are logged every `LOG_EVERY` steps.  Note that SFT reports a
**cross-entropy** training loss while the DPO variants report the **DPO loss** —
these are on different scales and are not directly comparable.  The
accuracy / precision / recall / F1 subplots use the same [0, 1] scale and
are directly comparable across all runs.

In [ ]:
TRAIN_METRICS = ['loss', 'accuracy', 'precision', 'recall', 'f1']
TRAIN_LABELS  = ['Loss', 'Accuracy', 'Precision\n(macro)', 'Recall\n(macro)', 'F1\n(macro)']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, metric, label in zip(axes, TRAIN_METRICS, TRAIN_LABELS):
    for run_name, log_dir in RUNS.items():
        # SFT logs 'train/loss'; DPO variants log 'train/dpo_loss'
        if metric == 'loss':
            tag = 'train/loss' if run_name == 'SFT' else 'train/dpo_loss'
        else:
            tag = f'train/{metric}'
        steps, values = load_scalars(log_dir, tag)
        ax.plot(steps, values, color=COLORS[run_name], alpha=0.85,
                linewidth=1.6, label=run_name)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Step')
    ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if metric != 'loss':
        ax.set_ylim(-0.05, 1.05)
    else:
        ax.set_ylabel('Loss (CE for SFT, DPO for others)', fontsize=8)

fig.legend(handles=legend_handles, loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.16), fontsize=10, frameon=True)
fig.suptitle('Training Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RUNS_DIR}/training_curves.png', bbox_inches='tight', dpi=150)
plt.show()

### 3.2 Validation Curves

Validation is run once per epoch (`VAL_EVERY = 1`) using the **full training set
as the in-context context** and the held-out validation split as query samples.
This mirrors the actual inference pattern of TabICL and gives a fair epoch-level
estimate of generalisation.  All five metrics share the same scale and are directly
comparable across methods.

In [ ]:
VAL_METRICS = ['loss', 'accuracy', 'precision', 'recall', 'f1']
VAL_LABELS  = ['Loss', 'Accuracy', 'Precision\n(macro)', 'Recall\n(macro)', 'F1\n(macro)']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, metric, label in zip(axes, VAL_METRICS, VAL_LABELS):
    for run_name, log_dir in RUNS.items():
        steps, values = load_scalars(log_dir, f'val/{metric}')
        ax.plot(steps, values, color=COLORS[run_name], alpha=0.85,
                linewidth=1.6, marker='o', markersize=4, label=run_name)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if metric != 'loss':
        ax.set_ylim(-0.05, 1.05)

fig.legend(handles=legend_handles, loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.16), fontsize=10, frameon=True)
fig.suptitle('Validation Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RUNS_DIR}/validation_curves.png', bbox_inches='tight', dpi=150)
plt.show()

### 4.3 Test Evaluation

We evaluate **seven models** on the held-out test set using the dataset that was saved
by `make_data.py` at the beginning of this notebook:

- **Vanilla TabICL** — the pretrained checkpoint with no fine-tuning, used as a
  zero-shot in-context baseline.
- **Neuralk NICL** — a competing pretrained tabular foundation model, evaluated
  via its scikit-learn-compatible `NICLClassifier` interface.
- **SFT / DPO variants** — the five fine-tuned checkpoints produced above.

The fine-tuned models use the full training split as in-context examples; vanilla
TabICL and Neuralk NICL do the same through their `fit` / `predict` interface.

In [ ]:
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report
)
from tabicl import TabICL, TabICLClassifier
from neuralk import NICLClassifier
from tabicl.finetune.utils import evaluate

# ── Load dataset saved by make_data.py ───────────────────────────────────────
data = np.load(DATASET_PATH)
X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val   = data['X_val'],   data['y_val']
X_test,  y_test  = data['X_test'],  data['y_test']
print(f'Dataset — train: {len(y_train)}, val: {len(y_val)}, test: {len(y_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Helper: evaluate a sklearn-style model ───────────────────────────────────
def eval_sklearn(preds):
    acc  = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='macro', zero_division=0)
    rec  = recall_score(y_test, preds, average='macro', zero_division=0)
    f1   = f1_score(y_test, preds, average='macro', zero_division=0)
    print('Classification report:\n', classification_report(y_test, preds, zero_division=0))
    print(f'  acc={acc:.4f}  prec={prec:.4f}  rec={rec:.4f}  f1={f1:.4f}')
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

test_results = {}

# ── Vanilla TabICL (pretrained, no fine-tuning) ───────────────────────────────
print()
print('=' * 60)
print('  Evaluating: Vanilla TabICL')
print('=' * 60)
tabicl_clf = TabICLClassifier()
tabicl_clf.fit(X_train, y_train)
test_results['Vanilla TabICL'] = eval_sklearn(tabicl_clf.predict(X_test))

# ── Neuralk NICLClassifier ────────────────────────────────────────────────────
print()
print('=' * 60)
print('  Evaluating: Neuralk NICL')
print('=' * 60)
nicl_clf = NICLClassifier()
nicl_clf.fit(X_train, y_train)
test_results['Neuralk NICL'] = eval_sklearn(nicl_clf.predict(X_test))

# ── Fine-tuned checkpoints ────────────────────────────────────────────────────
CKPT_PATHS = {
    'SFT':           f'{RUNS_DIR}/sft/tabicl_sft.ckpt',
    'DPO-random':    f'{RUNS_DIR}/dpo_random/tabicl_dpo.ckpt',
    'DPO-majority':  f'{RUNS_DIR}/dpo_majority/tabicl_dpo.ckpt',
    'DPO-hard':      f'{RUNS_DIR}/dpo_hard/tabicl_dpo.ckpt',
    'DPO-confusion': f'{RUNS_DIR}/dpo_confusion/tabicl_dpo.ckpt',
}

for run_name, ckpt_path in CKPT_PATHS.items():
    print()
    print('=' * 60)
    print(f'  Evaluating: {run_name}')
    print('=' * 60)
    ckpt  = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    model = TabICL(**ckpt['config'])
    model.load_state_dict(ckpt['state_dict'])
    model = model.to(device)
    loss, acc, prec, rec, f1 = evaluate(
        model, X_train, y_train, X_test, y_test, device
    )
    test_results[run_name] = {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1
    }
    print(f'  acc={acc:.4f}  prec={prec:.4f}  rec={rec:.4f}  f1={f1:.4f}')

print()
print('All evaluations complete.')

### 4.4 Test Performance — Bar Chart Comparison

The bar charts below summarise test-set performance of all **seven models** across
all four classification metrics.  The two pretrained baselines (Vanilla TabICL and
Neuralk NICL) appear first, followed by the five fine-tuned runs, making it easy to
see how much — and in which direction — fine-tuning moves each metric.

Key questions to look for:
- Does fine-tuning (SFT or DPO) improve over the vanilla pretrained baselines?
- Does DPO improve **macro precision / recall / F1** over SFT, even if accuracy stays similar?
- Which negative-sampling strategy best counteracts the imbalance bias?
- How does TabICL compare with Neuralk NICL on this imbalanced task?

In [ ]:
BAR_METRICS = ['accuracy', 'precision', 'recall', 'f1']
BAR_LABELS  = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)']

run_names  = list(ALL_COLORS.keys())
bar_colors = [ALL_COLORS[r] for r in run_names]
x          = np.arange(len(run_names))
bar_width  = 0.55

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, metric, label in zip(axes, BAR_METRICS, BAR_LABELS):
    values = [test_results.get(r, {}).get(metric, 0.0) for r in run_names]
    bars   = ax.bar(x, values, width=bar_width, color=bar_colors,
                    edgecolor='white', linewidth=0.8)
    ax.set_title(label, fontweight='bold', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(run_names, rotation=40, ha='right', fontsize=8)
    ax.set_ylim(0, 1.15)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # Add a vertical separator after the two pretrained baselines
    ax.axvline(x=1.5, color='#aaaaaa', linewidth=1.0, linestyle='--')
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.018,
            f'{val:.3f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold',
        )

fig.suptitle(
    'Test Set Performance — Pretrained Baselines vs Fine-Tuned TabICL',
    fontsize=14, fontweight='bold',
)
all_legend_handles = [
    mpatches.Patch(color=c, label=n) for n, c in ALL_COLORS.items()
]
fig.legend(handles=all_legend_handles, loc='lower center', ncol=7,
           bbox_to_anchor=(0.5, -0.16), fontsize=9, frameon=True)
plt.tight_layout()
plt.savefig(f'{RUNS_DIR}/test_barplots.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 5. Summary

This notebook showed how DPO-based fine-tuning can be applied to TabICL to improve
performance on imbalanced tabular classification, and benchmarked it against two
pretrained baselines.  Key takeaways:

- **Vanilla TabICL** provides a zero-shot in-context baseline without any fine-tuning.
- **Neuralk NICL** is a competing foundation model evaluated under the same conditions.
- **SFT** provides a solid accuracy baseline but may over-predict the majority class,
  resulting in poor macro-averaged precision/recall/F1.
- **DPO** introduces an explicit penalty for predicting the rejected label, which can
  shift probability mass toward minority classes and improve macro metrics even when
  accuracy stays similar or slightly decreases.
- The **negative-sampling strategy** matters: majority-class negatives directly target
  the dominant bias, hard negatives provide the strongest gradient signal, and
  confusion-set pairing uses dataset structure to guide the rejection.
- All hyperparameters (β, learning rate, epochs) and data parameters are fully
  controllable via the CLI, making it straightforward to run ablations.

Saved figures:
- `runs/demo/training_curves.png` — per-step training metric curves
- `runs/demo/validation_curves.png` — per-epoch validation metric curves
- `runs/demo/test_barplots.png` — held-out test performance comparison (all 7 models)